In [1]:
# ============================================================
# Logistic Regression Baseline
# ============================================================

# 1. Import libraries
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    log_loss,
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
    accuracy_score,
    f1_score,
    confusion_matrix,
    classification_report
)


# ============================================================
# 2. Configuration
# ============================================================

TARGET = "pres.abs"
SPECIES = "Species"

NUMERIC_FEATURES = [
    "disturb",
    "rainann",
    "soildepth",
    "soilfert",
    "tempann",
    "topo",
    "easting",
    "northing"
]

TRAIN_FILE = "train_split.csv"
TEST_FILE = "test_split.csv"

RANDOM_STATE = 42
THRESHOLD = 0.5


# ============================================================
# 3. Functions
# ============================================================

def load_data(train_file, test_file):
    """
    Load the prepared training and test datasets.
    """

    train_df = pd.read_csv(train_file)
    test_df = pd.read_csv(test_file)

    required_columns = {TARGET, SPECIES}

    for dataset_name, df in {
        "training": train_df,
        "test": test_df
    }.items():

        missing_columns = required_columns.difference(df.columns)

        if missing_columns:
            raise ValueError(
                f"{dataset_name} data is missing columns: "
                f"{sorted(missing_columns)}"
            )

    print("Data loaded successfully.")
    print(
        f"Training data: {train_df.shape[0]} rows, "
        f"{train_df.shape[1]} columns"
    )
    print(
        f"Test data: {test_df.shape[0]} rows, "
        f"{test_df.shape[1]} columns"
    )

    print(
        f"Training presence rate: "
        f"{train_df[TARGET].mean():.4f}"
    )
    print(
        f"Test presence rate: "
        f"{test_df[TARGET].mean():.4f}"
    )

    return train_df, test_df


def build_features(df, feature_columns=None):
    """
    Build numerical predictors and one-hot encoded species variables.

    When feature_columns is supplied, the resulting dataframe is aligned
    with the training feature columns.
    """

    numeric_present = [
        feature
        for feature in NUMERIC_FEATURES
        if feature in df.columns
    ]

    species_dummies = pd.get_dummies(
        df[SPECIES],
        prefix="sp",
        dtype=float
    )

    numeric_data = (
        df[numeric_present]
        .reset_index(drop=True)
        .astype(float)
    )

    species_dummies = species_dummies.reset_index(drop=True)

    X = pd.concat(
        [
            numeric_data,
            species_dummies
        ],
        axis=1
    )

    if feature_columns is not None:
        X = X.reindex(
            columns=feature_columns,
            fill_value=0
        )

    return X.astype(float)


def prepare_data(train_df, test_df):
    """
    Create the feature matrices, target vectors and standardised data.
    """

    X_train = build_features(train_df)

    feature_columns = X_train.columns.tolist()

    X_test = build_features(
        test_df,
        feature_columns=feature_columns
    )

    y_train = train_df[TARGET].astype(int)
    y_test = test_df[TARGET].astype(int)

    scaler = StandardScaler()

    X_train_scaled = scaler.fit_transform(X_train)

    # The test data use the mean and standard deviation learned
    # from the training data.
    X_test_scaled = scaler.transform(X_test)

    print("\nFeature preparation completed.")
    print(f"Number of features: {len(feature_columns)}")
    print("Feature names:")
    print(feature_columns)

    return {
        "X_train": X_train,
        "X_test": X_test,
        "X_train_scaled": X_train_scaled,
        "X_test_scaled": X_test_scaled,
        "y_train": y_train,
        "y_test": y_test,
        "feature_columns": feature_columns,
        "scaler": scaler
    }


def train_logistic_model(
    X_train,
    y_train,
    C=1.0,
    class_weight=None,
    penalty="l2",
    max_iter=2000
):
    """
    Train a Logistic Regression model.
    """

    model = LogisticRegression(
        C=C,
        penalty=penalty,
        class_weight=class_weight,
        max_iter=max_iter,
        random_state=RANDOM_STATE
    )

    model.fit(
        X_train,
        y_train
    )

    print("\nModel training completed.")

    return model


def evaluate_model(
    model,
    X_test,
    y_test,
    threshold=0.5
):
    """
    Evaluate the model using probability-based and classification metrics.
    """

    # Probability that pres.abs = 1
    test_prob = model.predict_proba(X_test)[:, 1]

    # Avoid probabilities being exactly 0 or 1 when calculating log loss
    test_prob = np.clip(
        test_prob,
        1e-15,
        1 - 1e-15
    )

    # Convert probabilities into class predictions
    test_pred = (
        test_prob >= threshold
    ).astype(int)

    results = {
        "log_loss": log_loss(
            y_test,
            test_prob
        ),
        "roc_auc": roc_auc_score(
            y_test,
            test_prob
        ),
        "pr_auc": average_precision_score(
            y_test,
            test_prob
        ),
        "brier_score": brier_score_loss(
            y_test,
            test_prob
        ),
        "accuracy": accuracy_score(
            y_test,
            test_pred
        ),
        "f1": f1_score(
            y_test,
            test_pred,
        )
    }

    print("\nTEST PERFORMANCE")
    print("=" * 45)

    print(f"Log loss:    {results['log_loss']:.4f}")
    print(f"ROC-AUC:     {results['roc_auc']:.4f}")
    print(f"PR-AUC:      {results['pr_auc']:.4f}")
    print(f"Brier score: {results['brier_score']:.4f}")
    print(f"Accuracy:    {results['accuracy']:.4f}")
    print(f"F1 score:    {results['f1']:.4f}")

    print("\nConfusion matrix:")
    print(confusion_matrix(y_test, test_pred))

    print("\nClassification report:")
    print(
        classification_report(
            y_test,
            test_pred
        )
    )

    return results, test_prob, test_pred


def get_coefficient_table(model, feature_columns):
    """
    Return Logistic Regression coefficients in a dataframe.
    """

    coefficient_table = pd.DataFrame({
        "feature": feature_columns,
        "coefficient": model.coef_[0]
    })

    coefficient_table["odds_ratio"] = np.exp(
        coefficient_table["coefficient"]
    )

    coefficient_table["absolute_coefficient"] = (
        coefficient_table["coefficient"].abs()
    )

    coefficient_table = coefficient_table.sort_values(
        by="absolute_coefficient",
        ascending=False
    )

    return coefficient_table


def run_baseline_experiment(
    train_file=TRAIN_FILE,
    test_file=TEST_FILE,
    C=1.0,
    class_weight=None,
    threshold=THRESHOLD
):
    """
    Run the full Logistic Regression baseline workflow.
    """

    train_df, test_df = load_data(
        train_file,
        test_file
    )

    prepared_data = prepare_data(
        train_df,
        test_df
    )

    model = train_logistic_model(
        X_train=prepared_data["X_train_scaled"],
        y_train=prepared_data["y_train"],
        C=C,
        class_weight=class_weight
    )

    results, test_prob, test_pred = evaluate_model(
        model=model,
        X_test=prepared_data["X_test_scaled"],
        y_test=prepared_data["y_test"],
        threshold=threshold
    )

    coefficient_table = get_coefficient_table(
        model,
        prepared_data["feature_columns"]
    )

    return {
        "model": model,
        "results": results,
        "test_probabilities": test_prob,
        "test_predictions": test_pred,
        "coefficient_table": coefficient_table,
        "scaler": prepared_data["scaler"],
        "feature_columns": prepared_data["feature_columns"],
        "train_df": train_df,
        "test_df": test_df
    }


# ============================================================
# 4. Run the baseline Logistic Regression
# ============================================================

baseline_result = run_baseline_experiment(
    C=1.0,
    class_weight=None,
    threshold=0.5
)



Data loaded successfully.
Training data: 4102 rows, 14 columns
Test data: 1026 rows, 14 columns
Training presence rate: 0.0631
Test presence rate: 0.0643

Feature preparation completed.
Number of features: 16
Feature names:
['disturb', 'rainann', 'soildepth', 'soilfert', 'tempann', 'topo', 'easting', 'northing', 'sp_Cacophis kreftii', 'sp_Calyptotis scutirostrum', 'sp_Coeranoscincus reticulatus', 'sp_Egernia mcpheei', 'sp_Eulamprus murrayi', 'sp_Ophioscincus truncatus', 'sp_Pseudechis porphyricaus', 'sp_Saltuarius swaini']

Model training completed.

TEST PERFORMANCE
Log loss:    0.1954
ROC-AUC:     0.8217
PR-AUC:      0.2403
Brier score: 0.0537
Accuracy:    0.9357
F1 score:    0.0000

Confusion matrix:
[[960   0]
 [ 66   0]]

Classification report:
              precision    recall  f1-score   support

           0       0.94      1.00      0.97       960
           1       0.00      0.00      0.00        66

    accuracy                           0.94      1026
   macro avg       0.4

C:\Users\13533\Anaconda3\lib\site-packages\sklearn\linear_model\least_angle.py:30: DeprecationWarning: `np.float` is a deprecated alias for the builtin `float`. To silence this warning, use `float` by itself. Doing this will not modify any behavior and is safe. If you specifically wanted the numpy scalar type, use `np.float64` here.
Deprecated in NumPy 1.20; for more details and guidance: https://numpy.org/devdocs/release/1.20.0-notes.html#deprecations
  method='lar', copy_X=True, eps=np.finfo(np.float).eps,
C:\Users\13533\Anaconda3\lib\site-packages\sklearn\linear_model\least_angle.py:167: DeprecationWarning: `np.float` is a deprecated alias for the builtin `float`. To silence this warning, use `float` by itself. Doing this will not modify any behavior and is safe. If you specifically wanted the numpy scalar type, use `np.float64` here.
Deprecated in NumPy 1.20; for more details and guidance: https://numpy.org/devdocs/release/1.20.0-notes.html#deprecations
  method='lar', copy_X=True,